# Repository edition: 02_causal_interventions

Run from this repository in your existing GPU environment. Source is in `../suites/02_causal_interventions/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Three Ways: complete causal-revision experiment runner

**Real data and pretrained models; no smoke-test mode.** This notebook embeds all code.
Upload this file to your Lambda machine and run the cells in order. No cohort file paths are required.

The default runs **logit origins and live repairs on Llama-3.2-1B**, on a fixed set of up to 600 PopQA evaluation prompts, plus independent calibration candidates. All eligible evaluation failures **and successes** receive repair sweeps. It also runs planted controls on up to 100 available successes.

Select `alignment`, `training`, or `full` below to run the larger campaigns. The full campaign includes many 7B/8B models and checkpoints: plan for substantial GPU time and disk use. FP32 is mandatory; 7B/8B weights alone require about 28–32 GB, plus working memory. An 80-GB GPU is the safest choice for those models.

This is a new fixed-cohort study, not automatic recovery of the original paper cohorts. Exact origin decomposition requires model passes; the prior results ZIP does not contain sufficient coefficient caches. See the bundled `README.txt` for all protocol details and departures from the draft proposal.

The code does **not** assume the predicted findings. Numerical failures stop the run. Missing donors and insufficient calibration remain explicit.

In [ ]:
# 1. Settings — edit before running the notebook.
PRESET = "one_model"  # "one_model", "alignment", "training", "full"
RUN_NAME = "causal_v1"
EVALUATION_PER_RELATION = 100
CALIBRATION_CANDIDATES_PER_RELATION = 250
CHAT_TEMPLATE_CHECK = False  # secondary shared-template check; doubles selected passes
TORCH_CUDA_WHEELS = "cu128"  # cu126 only for a driver compatible with CUDA 12.6, not 12.8
RUN_IN_BACKGROUND = False  # True launches a detached process; monitor with the later cell
# Optional configuration overrides. Default repairs are NOT limited to 60 examples.
OVERRIDES = {}
# Example: OVERRIDES = {"save_frozen_cache": True}  # potentially very large
# Exposure stratification needs actual corpus counts: set exposure_csv only if available.


## 2. Prepare the repository source
This writes into the current notebook directory. No existing project files are overwritten: each source bundle has a unique directory.

In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile
BUNDLE = _source_bundle('02_causal_interventions')
EXPECTED_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('02_causal_interventions'))).hexdigest()
raw = base64.b64decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256
WORK = pathlib.Path.cwd().resolve()
CODE = WORK / ("threeways_causal_code_" + EXPECTED_SHA256[:12])
CODE.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target = (CODE / name).resolve()
        if CODE.resolve() not in target.parents:
            raise RuntimeError("Unsafe archive entry")
        if target.exists() and target.is_file() and target.read_bytes() != z.read(name):
            raise RuntimeError("Local source was edited; preserve your edits and choose a new source directory.")
    z.extractall(CODE)
print("Source:", CODE)
print("Protocol guide:", CODE / "README.txt")


## 3. Create the isolated environment
This may take several minutes on the first run. Experiments run using this environment's Python, even if your notebook uses a different kernel. The setup also registers **Python (Three Ways Causal)** for future notebooks. No kernel restart is needed for this runner.

If Python reports that `venv` is missing, install `python3-venv` using your Lambda terminal, then rerun this cell. If CUDA is unavailable, resolve the driver/wheel mismatch rather than falling back silently to CPU.

In [ ]:
VENV = WORK / "threeways_causal_env"
PYTHON = VENV / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
marker = VENV / "installed_protocol.json"
expected = {"bundle": EXPECTED_SHA256, "cuda_wheels": TORCH_CUDA_WHEELS}
if not marker.exists() or json.loads(marker.read_text()) != expected:
# Use the existing environment; dependencies are documented in requirements.txt.
# Use the existing environment; dependencies are documented in requirements.txt.
# Use the existing environment; dependencies are documented in requirements.txt.
    subprocess.run([str(PYTHON), "-m", "ipykernel", "install", "--user", "--name", "threeways-causal",
                    "--display-name", "Python (Three Ways Causal)"], check=True)
    marker.write_text(json.dumps(expected))
subprocess.run([str(PYTHON), "-c",
    "import torch,transformers; assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print('GPU:',torch.cuda.get_device_name(0)); print('Torch:',torch.__version__); "
    "print('Transformers:',transformers.__version__); "
    "print('Total VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,1))"], check=True)


## 4. Build and inspect the run configuration
`full` selects the alignment families, Pythia revisions and discovered OLMo-2 1B/7B stage-1 checkpoints, with repairs on six named core models. Model and dataset revisions are pinned when the run begins. Hard decoys are chosen by the first member of each family and reused.

Changing the configuration or code requires a **new RUN_NAME**. You may also edit the generated configuration before launching, but preserve the filename used below.

In [ ]:
config_text = subprocess.check_output([str(PYTHON), "-c",
    "import json,sys; from causal_suite.config import preset; print(json.dumps(preset(sys.argv[1])))", PRESET],
    cwd=CODE, text=True)
CONFIG = json.loads(config_text)
CONFIG.update(run_name=RUN_NAME, evaluation_per_relation=EVALUATION_PER_RELATION,
              calibration_candidates_per_relation=CALIBRATION_CANDIDATES_PER_RELATION,
              chat_check=CHAT_TEMPLATE_CHECK)
CONFIG.update(OVERRIDES)
RESULTS = WORK / "threeways_causal_results" / RUN_NAME
CONFIG_PATH = WORK / (RUN_NAME + "_config.json")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))
print("Results:", RESULTS)


## 5. Hugging Face access
Meta models require approved model access. A token is passed only through the process environment and is not written to the run configuration. Leave blank to use credentials already saved on the machine. Ungated Qwen/OLMo/Pythia presets do not require a Meta token.

In [ ]:
from getpass import getpass
if any(m["repo"].startswith("meta-llama/") for m in CONFIG["models"]) and not os.environ.get("HF_TOKEN"):
    token = getpass("Hugging Face token (Enter to use existing login): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
    del token


## 6. Run implementation checks
Small randomly initialized models test accounting, all four architecture adapters, live interventions, conservation, statistics, and report generation. These tests are not scientific experiments and do not replace the real run below.

In [ ]:
subprocess.run([str(PYTHON), "-m", "pytest", "tests", "-q"], cwd=CODE, check=True)


## 7. Run the full selected experiment
Each example is saved atomically. Rerun this cell with identical settings to resume.

**Do not launch two workers into the same results directory.** This cell checks for a running PID recorded by this notebook. The default waits for completion and streams progress. With `RUN_IN_BACKGROUND=True`, it returns immediately; the process continues after disconnecting from Jupyter. Later cells show partial results until the worker completes.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
PID_FILE = RESULTS / "notebook_worker.pid"
LOG = RESULTS / "execution.log"
if PID_FILE.exists():
    old_pid = int(PID_FILE.read_text())
    try:
        os.kill(old_pid, 0)
    except ProcessLookupError:
        PID_FILE.unlink()
    else:
        raise RuntimeError(f"Process {old_pid} may still be running. Check it before launching another worker.")
(RESULTS / "exit_code.txt").unlink(missing_ok=True)
command = [str(PYTHON), "-u", "-m", "causal_suite.run", "--config", str(CONFIG_PATH), "--out", str(RESULTS)]
if RUN_IN_BACKGROUND:
    # A small supervisor records completion and removes its own PID marker.
    supervisor = """import subprocess,sys,pathlib
pidfile=pathlib.Path(sys.argv[1]); status=pathlib.Path(sys.argv[2])
try:
    code=subprocess.call(sys.argv[3:]);status.write_text(str(code))
finally:
    pidfile.unlink(missing_ok=True)
sys.exit(code)
"""
    with LOG.open("a") as log:
        worker = subprocess.Popen([str(PYTHON), "-c", supervisor, str(PID_FILE),
            str(RESULTS / "exit_code.txt"), *command], cwd=CODE, stdout=log, stderr=subprocess.STDOUT,
            start_new_session=True)
    PID_FILE.write_text(str(worker.pid))
    print("Launched process", worker.pid, "Log:", LOG)
else:
    worker = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, start_new_session=True)
    PID_FILE.write_text(str(worker.pid))
    try:
        with LOG.open("a") as log:
            for line in worker.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
        returncode = worker.wait()
        (RESULTS / "exit_code.txt").write_text(str(returncode))
        if returncode:
            raise RuntimeError(f"Run stopped with exit code {returncode}. Inspect {LOG}; completed examples are saved.")
    finally:
        if worker.poll() is not None:
            PID_FILE.unlink(missing_ok=True)


## 8. Monitor or regenerate reports
For background runs, rerun the first cell here to see progress. A zero exit code indicates completion; a nonzero code requires checking the log. Reporting can run separately on CPU after measurements exist.

In [ ]:
if LOG.exists():
    from collections import deque
    with LOG.open() as f:
        print("".join(deque(f, maxlen=25)))
status_path = RESULTS / "exit_code.txt"
print("Exit code:", status_path.read_text() if status_path.exists() else "not recorded yet")
print("Completed measurement sets:", len(list(RESULTS.glob("*/*/measurements.json"))))


In [ ]:
# Set True if you want to rebuild summaries from existing measurement/repair files.
REBUILD_REPORTS = False
if REBUILD_REPORTS:
    subprocess.run([str(PYTHON), "-m", "causal_suite.run", "--config", str(CONFIG_PATH),
                    "--out", str(RESULTS), "--report-only"], cwd=CODE, check=True)


## 9. Inspect and download
Start with coverage, donor availability, certificates and calibration sizes. Read `interpretation.json` before interpreting the figures. Native repair doses have different units; the notebook does not compute a misleading cross-family dose ratio. The common-cost analysis is explicitly exploratory.

Exposure analysis runs only when actual corpus counts are supplied. The source package includes `python -m causal_suite.exposure` to count local corpus JSONL documents. Corpus totals do not establish checkpoint-specific training exposure.

In [ ]:
from IPython.display import FileLink, display
REPORTS = RESULTS / "reports"
if REPORTS.exists():
    for name in ["coverage.csv", "origin_auc_holm.csv", "alignment_fix_odds_holm.csv",
                 "training_summary.csv", "repair_matrix_nine_control_tests.csv", "donor_coverage.csv"]:
        path = REPORTS / name
        if path.exists():
            print("\n" + name)
            print("\n".join(path.read_text().splitlines()[:8]))
    print("PDF figures:")
    for path in sorted(REPORTS.glob("*.pdf")):
        display(FileLink(str(path)))
else:
    print("No reports yet; check the running process or execution log.")


In [ ]:
# Compact archive excludes large optional tensor caches and model weights.
ARCHIVE = WORK / (RUN_NAME + "_results.zip")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file() and "frozen_cache" not in path.parts and path.suffix != ".pt":
            z.write(path, path.relative_to(RESULTS))
    z.write(CONFIG_PATH, "configuration.json")
    z.write(CODE / "README.txt", "PROTOCOL_README.txt")
    # Embed source in the results for exact auditability.
    for path in sorted((CODE / "causal_suite").glob("*.py")):
        z.write(path, "source/" + path.name)
print("Saved:", ARCHIVE)
if PID_FILE.exists():
    print("Worker may still be active: this archive is a PARTIAL snapshot; rerun this cell after completion.")
display(FileLink(str(ARCHIVE)))
